# 🦙 Llama-3 Fine-Tuning for E-Commerce Copywriting

This notebook fine-tunes **Llama-3-8B-Instruct** on our locally generated synthetic e-commerce dataset.

**Architecture: Dual-Mode (Resilient)**
- **Plan A (Unsloth):** 2x faster training, 70% less VRAM — activates if Unsloth is available
- **Plan B (HuggingFace Fallback):** VRAM-safe mode with adjusted batch params — activates automatically on failure
- Trains with LoRA (Low-Rank Adaptation) — only ~1% of model weights updated
- Exports to **GGUF format** for local Ollama deployment

**Steps:**
1. GPU Setup & Dependencies
2. Upload Dataset (Kaggle)
3. Dual-Mode Model Load + Train
4. Test Fine-Tuned Model
5. Export to GGUF for Ollama

## Step 1: GPU Setup & Install Dependencies

In [ ]:
import os
# Force single GPU — prevents T4 x2 multi-GPU tensor device mismatch errors
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("✅ Single GPU mode enabled (CUDA_VISIBLE_DEVICES=0)")

In [ ]:
%%capture
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install datasets

## Step 2: Upload Dataset (Kaggle)

Upload `synthetic_train_ollama.jsonl` via the right panel **Upload** button, then run this cell to locate it.

In [ ]:
import os, shutil

# Kaggle: search for uploaded JSONL file under /kaggle/input/
input_dir = "/kaggle/input"
jsonl_files = []

for root, dirs, files in os.walk(input_dir):
    for file in files:
        if file.endswith(".jsonl"):
            jsonl_files.append(os.path.join(root, file))

if jsonl_files:
    src = jsonl_files[0]
    dst = "synthetic_train_ollama.jsonl"
    shutil.copy(src, dst)
    print(f"✅ Dataset found and copied: {src}")
    print(f"✅ Size: {os.path.getsize(dst) / 1024 / 1024:.1f} MB")
else:
    print("❌ No JSONL file found! Upload it via the right panel → Upload.")

## Step 3: Dual-Mode Model Load + Train 🚀

**Plan A (Unsloth):** Maximum performance — 2x speed, 70% less VRAM

**Plan B (HuggingFace):** Auto-activated if Unsloth fails — VRAM-safe parameters

Effective batch size is mathematically identical in both modes: `2×4 = 1×8 = 8`

In [ ]:
# CRITICAL: Import unsloth first to apply optimizations globally before transformers loads
try:
    import unsloth
except ImportError:
    pass

import os, shutil, torch
from datasets import load_dataset
from transformers import TrainingArguments

# ── Config ───────────────────────────────────────────────────────────────────
max_seq_length = 2048
dataset_path   = "synthetic_train_ollama.jsonl"
UNSLOTH_MODEL  = "unsloth/llama-3-8b-Instruct-bnb-4bit"   # Unsloth pre-quantized
HF_MODEL       = "meta-llama/Meta-Llama-3-8B-Instruct"    # Standard HF fallback

raw_dataset = load_dataset("json", data_files=dataset_path, split="train")
print(f"Dataset loaded: {len(raw_dataset)} examples")

# ── State ─────────────────────────────────────────────────────────────────────
USE_UNSLOTH    = False
model          = None
tokenizer      = None
train_batch_size  = 1   # Conservative defaults (fallback)
grad_accum_steps  = 8

# ── PLAN A: UNSLOTH ───────────────────────────────────────────────────────────
try:
    from unsloth import FastLanguageModel

    print("🚀 Unsloth found — High-performance mode initializing...")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name    = UNSLOTH_MODEL,
        max_seq_length= max_seq_length,
        dtype         = None,
        load_in_4bit  = True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r                        = 16,
        target_modules           = ["q_proj","k_proj","v_proj","o_proj",
                                     "gate_proj","up_proj","down_proj"],
        lora_alpha               = 16,
        lora_dropout             = 0,
        bias                     = "none",
        use_gradient_checkpointing = "unsloth",  # Unsloth custom OOM protection
        random_state             = 3407,
        use_rslora               = False,
        loftq_config             = None,
    )
    USE_UNSLOTH      = True
    train_batch_size = 2   # Safe with Unsloth VRAM savings
    grad_accum_steps = 4   # Effective batch: 2×4 = 8 (same as fallback)
    print("✅ Unsloth configuration complete.")

# ── PLAN B: HUGGINGFACE SAFE MODE ────────────────────────────────────────────
except Exception as e:  # Catches ImportError, AttributeError, AcceleratorError…
    print(f"⚠️  Unsloth failed [{type(e).__name__}]: {e}")
    print("🛡️  Fallback active — HuggingFace Safe Mode, adjusting VRAM params...")

    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    bnb_config = BitsAndBytesConfig(
        load_in_4bit          = True,
        bnb_4bit_use_double_quant = True,
        bnb_4bit_quant_type   = "nf4",
        bnb_4bit_compute_dtype= torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)
    
    model = AutoModelForCausalLM.from_pretrained(
        HF_MODEL, quantization_config=bnb_config, device_map="auto"
    )
    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()  # Manual OOM protection

    lora_config = LoraConfig(
        r              = 16,
        lora_alpha     = 16,
        target_modules = ["q_proj","k_proj","v_proj","o_proj",
                          "gate_proj","up_proj","down_proj"],
        lora_dropout   = 0.05,
        bias           = "none",
        task_type      = "CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)

    # CRITICAL: reduced batch size to prevent OOM on 16GB T4
    # Effective batch remains identical: 1×8 = 2×4 = 8 ✓
    train_batch_size = 1
    grad_accum_steps = 8
    print("✅ HuggingFace Safe Mode configuration complete.")

# ── DATASET FORMATTING ───────────────────────────────────────────────────────
print("⏳ Formatting dataset with tokenizer chat template...")
from unsloth.chat_templates import get_chat_template

# Ensure tokenizer has the correct chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3",
    mapping={"role": "role", "content": "content", "user": "user", "assistant": "assistant"}
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = raw_dataset.map(formatting_prompts_func, batched=True)
print("✅ Dataset formatting complete. Ready for training.")

# ── CLEAN OLD CACHE ───────────────────────────────────────────────────────────
if os.path.exists("/kaggle/working/unsloth_compiled_cache"):
    shutil.rmtree("/kaggle/working/unsloth_compiled_cache")
    print("✅ Old Unsloth cache cleared.")

# ── SHARED TRAINING LOOP ──────────────────────────────────────────────────────
from trl import SFTTrainer

trainer = SFTTrainer(
    model             = model,
    tokenizer         = tokenizer,
    train_dataset     = dataset,
    dataset_text_field= "text",  # Now this works because we created the 'text' column!
    max_seq_length    = max_seq_length,
    dataset_num_proc  = 2,
    packing           = False,
    args = TrainingArguments(
        per_device_train_batch_size = train_batch_size,
        gradient_accumulation_steps = grad_accum_steps,
        warmup_steps                = 10,
        num_train_epochs            = 1,
        learning_rate               = 2e-4,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 1,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 3407,
        output_dir                  = "outputs",
        report_to                   = "none",
        save_strategy               = "no",  # Prevents Kaggle PicklingError
    ),
)

print(f"\n⚙️  Mode      : {'UNSLOTH (Max Speed)' if USE_UNSLOTH else 'HUGGINGFACE (VRAM Safe)'}")
print(f"📊  Effective Batch: {train_batch_size} × {grad_accum_steps} = {train_batch_size * grad_accum_steps}")
print("🏋️  Training started...\n")

trainer_stats = trainer.train()

print(f"\n✅ Training complete!")
print(f"   Final loss  : {trainer_stats.training_loss:.4f}")
print(f"   Total steps : {trainer_stats.global_step}")
print(f"   Duration    : {trainer_stats.metrics['train_runtime']:.0f} seconds")

## Step 4: Test the Fine-Tuned Model 🧪

Compare the fine-tuned model output on a sample product.

In [ ]:
# Switch to inference mode
if USE_UNSLOTH:
    FastLanguageModel.for_inference(model)

test_product = """Title: Neutrogena Hydro Boost Water Gel, 1.7 oz

Features:
- Oil-free face moisturizer with hyaluronic acid
- Instantly quenches dry skin
- Non-comedogenic, dye-free
- For extra-dry skin"""

messages = [{"role": "user", "content": test_product}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

print("=" * 60)
print("FINE-TUNED MODEL OUTPUT:")
print("=" * 60)
output = model.generate(
    input_ids     = inputs,
    max_new_tokens= 512,
    temperature   = 0.7,
    streamer      = text_streamer,
    use_cache     = True,
)

## Step 5: Export to GGUF (for Ollama) 📦

Converts the fine-tuned model to GGUF format for local Ollama deployment.

In [ ]:
# Save LoRA weights first
model.save_pretrained("finetuned_llama3_lora")
tokenizer.save_pretrained("finetuned_llama3_lora")
print("LoRA weights saved to finetuned_llama3_lora/")

# Export as GGUF Q4_K_M (best quality/size balance for Ollama)
if USE_UNSLOTH:
    model.save_pretrained_gguf(
        "finetuned_llama3_gguf",
        tokenizer,
        quantization_method="q4_k_m",
    )
    print("✅ GGUF export complete! → finetuned_llama3_gguf/")
else:
    print("ℹ️  GGUF export requires Unsloth. Run in Unsloth mode for GGUF output.")
    print("   You can still use the LoRA weights from finetuned_llama3_lora/")

In [ ]:
# List output files for download (Kaggle: copy path and use right panel)
import glob

gguf_files = glob.glob("finetuned_llama3_gguf/*.gguf")
lora_files = glob.glob("finetuned_llama3_lora/*")

if gguf_files:
    print(f"📦 GGUF file ready: {gguf_files[0]}")
    print("   → In Kaggle: right panel → Output → download")
else:
    print("📦 LoRA files ready:")
    for f in lora_files:
        print(f"   {f}")

## 🎉 Done!

### How to use the GGUF in Ollama:
1. Download the `.gguf` file from Kaggle Output panel
2. Create a `Modelfile`:
```
FROM ./finetuned_llama3.gguf
```
3. Run in terminal:
```bash
ollama create ecommerce-copywriter -f Modelfile
ollama run ecommerce-copywriter
```